# 01 — Preparação da coleção LExR

Este notebook organiza as etapas de preparação para o projeto.

O fluxo reproduzido aqui inclui:

1. configuração genérica dos caminhos;
2. inspeção dos arquivos de autores, documentos e qrels;
3. filtragem de `documents.json` para os autores presentes em `author_unique_prof-qrels.json`;
4. geração de `documents_ft.json`;
5. geração de `perfis_textuais_ft.json` a partir de título, palavras-chave e resumo;
6. verificação resumida dos arquivos produzidos.

A lógica reutilizável fica em `src/preprocessing/prepare_lexr.py`. Este notebook serve como registro executável do experimento de pré-processamento.


## 1. Configuração do projeto

O notebook não depende de caminhos específicos do Google Drive ou do ambiente original.

Apenas informe em `SOURCE_DIR` o diretório em que estão os arquivos originais da coleção LExR. As saídas são gravadas, por padrão, em `data/processed/` dentro do repositório.


In [ ]:
from pathlib import Path
import json
import sys

# Localiza a raiz do repositório procurando a pasta "src".
current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src").exists()), current)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.prepare_lexr import (
    build_ft_text_profiles,
    filter_documents,
    inspect_documents_jsonl,
    inspect_ids,
    inspect_qrels,
)

print(f"Raiz do projeto: {PROJECT_ROOT}")


## 2. Caminhos dos arquivos

Altere somente `SOURCE_DIR` para apontar para a pasta que contém os arquivos LExR utilizados no experimento.


In [ ]:
# Diretório externo que contém os arquivos da coleção LExR.
# Exemplo:
# SOURCE_DIR = Path("/caminho/para/lexr")
SOURCE_DIR = Path("CAMINHO_PARA_OS_ARQUIVOS_LEXR")

# Saídas processadas do projeto.
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENTS_RAW = SOURCE_DIR / "documents.json"
AUTHOR_IDS = SOURCE_DIR / "author_unique_prof-qrels.json"
QRELS = SOURCE_DIR / "LExR-prof-qrels"

# Arquivos já existentes no estudo e úteis para conferência, quando disponíveis.
FILTERED_DOCUMENTS = SOURCE_DIR / "filtered_documents.json"
FILTERED_AUTHOR_IDS = SOURCE_DIR / "author_unique_prof-qrels_filtrado.json"
FILTERED_QRELS = SOURCE_DIR / "LExR-prof-qrels_filtrado"

# Arquivos gerados neste notebook.
DOCUMENTS_FT = OUTPUT_DIR / "documents_ft.json"
PROFILES_FT = OUTPUT_DIR / "perfis_textuais_ft.json"

paths = {
    "documents.json": DOCUMENTS_RAW,
    "author_unique_prof-qrels.json": AUTHOR_IDS,
    "LExR-prof-qrels": QRELS,
    "filtered_documents.json": FILTERED_DOCUMENTS,
    "author_unique_prof-qrels_filtrado.json": FILTERED_AUTHOR_IDS,
    "LExR-prof-qrels_filtrado": FILTERED_QRELS,
}

for name, path in paths.items():
    status = "OK" if path.exists() else "não encontrado"
    print(f"{name:40s} -> {status}")


## 3. Inspeção das entradas

A inspeção abaixo serve para observar o comportamento exploratório dos dados.
- arquivos `author_unique_*.json` são tratados como listas de IDs;
- `documents.json` e `filtered_documents.json` são tratados como JSON Lines;
- os qrels seguem o formato `autor_id  iteração  tag  relevância`, preservando tags com espaços.


In [ ]:
# Arquivo de autores usado na preparação para fine-tuning.
inspect_ids(AUTHOR_IDS, examples=3)

# Julgamentos de relevância originais.
inspect_qrels(QRELS, examples=3)

# Inspeção em streaming do arquivo de documentos.
inspect_documents_jsonl(DOCUMENTS_RAW, examples=2)


## 4. Geração de `documents_ft.json`

A etapa mantém somente as publicações de `documents.json` que possuem pelo menos um autor presente em `author_unique_prof-qrels.json`.

A publicação é mantida integralmente, sem alteração dos demais campos. A leitura é feita em streaming para evitar carregar toda a coleção em memória.


In [ ]:
summary_documents_ft = filter_documents(
    documents_path=DOCUMENTS_RAW,
    author_ids_path=AUTHOR_IDS,
    output_path=DOCUMENTS_FT,
)

summary_documents_ft

## 5. Geração de `perfis_textuais_ft.json`

Para cada pesquisador, as publicações selecionadas são convertidas em representações textuais utilizando, nesta ordem:

- título;
- palavras-chave;
- resumo.

O procedimento preserva a ordem em que as publicações aparecem no arquivo de entrada. Nesta etapa não é aplicado limite de 50 publicações e não é realizada ordenação adicional por ano.


In [ ]:
summary_profiles_ft = build_ft_text_profiles(
    documents_path=DOCUMENTS_FT,
    author_ids_path=AUTHOR_IDS,
    output_path=PROFILES_FT,
)

summary_profiles_ft


## 6. Verificação das saídas

A célula abaixo mostra apenas uma pequena amostra do arquivo de perfis para evitar imprimir todo o conteúdo.


In [ ]:
with PROFILES_FT.open("r", encoding="utf-8") as f:
    profiles = json.load(f)

print(f"Total de autores no arquivo: {len(profiles):,}")

for author_id, profile in list(profiles.items())[:3]:
    print("\n" + "=" * 80)
    print(f"Autor: {author_id}")
    print(f"Número de publicações: {profile.get('n_publicacoes', 0)}")
    publications = profile.get("publicacoes_textuais", [])
    if publications:
        print("\nPrimeira publicação textual:")
        print(publications[0][:1000])


## 7. Arquivos produzidos

Ao final da execução, este notebook gera:

- `documents_ft.json`
- `perfis_textuais_ft.json`

Esses arquivos correspondem às etapas presentes nos notebooks originais de preparação para o ajuste fino.

As etapas de construção dos perfis específicos dos baselines e dos modelos Qwen são executadas nos notebooks seguintes de `experiments/01_preprocessing/`.
